<a href="https://colab.research.google.com/github/RezaBahani/IUTComputationalPhysics/blob/main/Reza_bahani_final_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install pysr

In [ ]:
pip install pysindy

In [ ]:
# Esential imprts
import torch
import torch.nn as nn
import torch.optim as optim
from torch.autograd import grad
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
import pandas as pd
import seaborn as sns
import pysr
import pysindy as ps
is_available = torch.cuda.is_available()
print(f"Is CUDA available? {is_available}")
# If it is available, print details.
if is_available:
    print(f"Number of GPUs: {torch.cuda.device_count()}")
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")



In [ ]:
#0. Reproducibility
torch.manual_seed(42)
np.random.seed(42)

#1. Neural Network Architecture
class PhysicsInformedNN(nn.Module):
    def __init__(self,input_dim=2,output_dim=2,hidden_dim=50):
        super(PhysicsInformedNN, self).__init__()
        self.fc = nn.Sequential(
            nn.Linear(input_dim, hidden_dim), nn.Tanh(),
            nn.Linear(hidden_dim, hidden_dim), nn.Tanh(),
            nn.Linear(hidden_dim, hidden_dim), nn.Tanh(),
            nn.Linear(hidden_dim, output_dim)
        )

    def forward(self, x, t):
        xt = torch.cat([x, t], dim = 1)
        out = self.fc(xt)
        return out[:, 0:1], out[:, 1:2]


#2. Physics Operator Impelementation
def apply_sqrt_operator_fft(psi_real, psi_imag, x_domain, hbar, m, c):
    '''Apllies the salpeter equation operator : sqrt(m^2 c^4 - c^2 hbar^2 del^2)'''
    if psi_real.ndim == 1:
        psi_real = psi_real.unsqueeze(0)
    if psi_imag.ndim == 1:
        psi_imag = psi_imag.unsqueeze(0)

    N_x = psi_real.shape[1]
    L = x_domain[1] - x_domain[0]
    psi_complex = torch.complex(psi_real, psi_imag)
    psi_fft = torch.fft.fft(psi_complex, dim = 1, norm = 'ortho')
    k = 2 * np.pi * torch.fft.fftfreq(N_x, d = L/N_x).to(psi_real.device)
    k = k.unsqueeze(0)
    sqrt_op_k = torch.sqrt(m**2 * c**4 + c**2 * hbar**2 * k**2 + 1e-8)
    result_fft = sqrt_op_k * psi_fft
    result_complex = torch.fft.ifft(result_fft, dim = 1, norm = 'ortho')
    return result_complex.real, result_complex.imag


#3. Unit Test For the Physics Operator
def test_operator_accuracy():
    "Using Plane Wave for the Test"
    test_k = 5.0

    # A plane wave that is the eigenfunction of H: H * exp(i * k * x) = E * exp(i * k * x)
    test_psi_real = torch.cos(test_k * x_grid.squeeze())
    test_psi_imag = torch.sin(test_k * x_grid.squeeze())

    energy_val = m**2 * c**4 + c**2 * hbar**2 * test_k**2
    expected_energy = torch.sqrt(torch.tensor(energy_val))
    expected_Hpsi_real = expected_energy * test_psi_real

    # Actual result of the operator
    actual_Hpsi_real, _ = apply_sqrt_operator_fft(
        test_psi_real.view(1, -1), test_psi_imag.view(1, -1), x_domain, hbar, m, c
    )

    # Compare the results
    error = torch.mean((expected_Hpsi_real - actual_Hpsi_real.squeeze())**2)
    if error < 1e-5:
        print(f"Unit Test Passed. L2 Error: {error.item(): .2e}\n")
    else:
        print(f"Unit Test Failed. L2 Error: {error.item(): .2e}\n")


#4. Loss Function Definitions
def pde_loss(t, model):
    X = x_grid.repeat(t.shape[0], 1).requires_grad_(True)
    T = t.repeat(1, N_x).view(-1, 1).requires_grad_(True)
    psi_real, psi_imag = model(X,T)
    d_real_dt, d_imag_dt = grad(psi_real.sum(), T, create_graph = True)[0], grad(psi_imag.sum(), T, create_graph = True)[0]
    d_real_dt, d_imag_dt = d_real_dt.view(t.shape[0], N_x), d_imag_dt.view(t.shape[0], N_x)
    LHS_real, LHS_imag = -hbar * d_imag_dt, hbar * d_real_dt
    psi_real_grid, psi_imag_grid = psi_real.view(t.shape[0], N_x), psi_imag.view(t.shape[0], N_x)
    RHS_real, RHS_imag = apply_sqrt_operator_fft(psi_real_grid, psi_imag_grid, x_domain, hbar, m, c)
    return torch.mean((LHS_real - RHS_real)**2) + torch.mean((LHS_imag - RHS_imag)**2)

def ic_loss(model, x_ic):
    t_ic = torch.zeros_like(x_ic).to(x_ic.device)
    psi_real, psi_imag = model(x_ic, t_ic)
    x0, sigma, k0 = -5.0, 1.0, 5.0
    norm = torch.exp(-(x_ic -x0)**2 / (2 * sigma**2))
    target_real, target_imag = norm * torch.cos(k0 * x_ic), norm * torch.sin(k0 * x_ic)
    return torch.mean((psi_real - target_real)**2 + (psi_imag - target_imag)**2)

def boundary_loss(model):
    device = next(model.parameters()).device
    x_boundary = torch.tensor([[x_domain[0]], [x_domain[1]]]).to(device)
    t_sample = torch.rand(100, 1).to(device) * 2.0

    t_boundary = t_sample.repeat(2, 1)
    x_boundary_repeated = x_boundary.repeat_interleave(100, dim = 0)

    psi_r_bc, psi_i_bc = model(x_boundary_repeated, t_boundary)

    return torch.mean(psi_r_bc**2 + psi_i_bc**2)

def norm_loss(model, t):
    "Put constrain that suggest probability is conserve"
    X = x_grid.repeat(t.shape[0], 1)
    T = t.repeat(1, N_x).view(-1, 1)

    psi_real, psi_imag = model(X, T)
    psi_real = psi_real.view(t.shape[0], N_x)
    psi_imag = psi_imag.view(t.shape[0], N_x)

    prob_density = psi_real**2 + psi_imag**2
    total_prob = torch.trapezoid(prob_density, x_grid.squeeze(), dim = 1)

    return torch.mean((total_prob -1.0)**2)


#5. Energy Monitoring Function
def compute_energy(psi_real, psi_imag):
    psi_real, psi_imag = psi_real.view(1, -1), psi_imag.view(1, -1)
    Hpsi_real, Hpsi_imag = apply_sqrt_operator_fft(psi_real, psi_imag, x_domain, hbar, m, c)
    integrad = psi_real * Hpsi_real + psi_imag * Hpsi_imag
    return torch.trapezoid(integrad, x_grid.squeeze()).item()


#6. Initialization and Setup

# Add to select the GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

hbar, c, m = 1.0, 1.0, 1.0
x_domain, N_x = (-15.0, 15,0), 1024

x_grid = torch.linspace(x_domain[0], x_domain[1], N_x).view(-1, 1).to(device)
model = PhysicsInformedNN()
model.to(device)

optimizer = optim.Adam(model.parameters(), lr = 1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience = 100, factor = 0.5)


#7. Run Pre-traning Validation
test_operator_accuracy()


#8. Main Trining Loop
epochs = 1000
energy_history = []
print('Starting Training...')

for epoch in range(epochs):
    optimizer.zero_grad()
    t_train = torch.FloatTensor(256, 1).uniform_(0, 2).to(device)

    # Calculating loss functions
    loss_pde_val = pde_loss(t_train, model)
    loss_ic_val = ic_loss(model, x_grid)
    loss_bc_val = boundary_loss(model)
    loss_norm_val = norm_loss(model, t_train)

    # Combine losses with appropriate weight
    loss = 1000.0 * loss_pde_val + 10.0 * loss_ic_val + 0.1 * loss_bc_val + 1000.0 * loss_norm_val
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm = 1.0)
    optimizer.step()
    scheduler.step(loss)

    if epoch % 200 == 0:
        with torch.no_grad():
            time_monitor = torch.full_like(x_grid, 1.5).to(device)
            psi_r_monitor, psi_i_monitor = model(x_grid, time_monitor)
            current_energy = compute_energy(psi_r_monitor, psi_i_monitor)
            energy_history.append(current_energy)
        print(f"Epoch: {epoch}/{epochs}, Loss: {loss.item(): .3e}, PDE: {loss_pde_val.item(): .3e}, "
              f"IC: {loss_ic_val.item(): .3e}, Norm: {loss_norm_val.item(): .3e}, Energy: {current_energy: .4f}")
print("\nTraining completed.")


#9. Visualization
print("\nGenerating visualization...")
plt.figure(figsize = (18, 5))
x_test = torch.linspace(x_domain[0], x_domain[1], 400).view(-1, 1).to(device)
t_test = torch.linspace(0, 2, 100).to(device)

X, T_vis = torch.meshgrid(x_test.squeeze(), t_test, indexing = 'xy')
X_flat, T_flat = X.reshape(-1, 1), T_vis.reshape(-1, 1)

with torch.no_grad():
    psi_real_flat, psi_imag_flat = model(X_flat, T_flat)
    prob_density = (psi_real_flat**2 + psi_imag_flat**2).reshape(X.shape)

T_vis_cpu = T_vis.cpu()
X_cpu = X.cpu()
prob_density_cpu = prob_density.cpu()
t_test_cpu = t_test.cpu()
x_test_cpu = x_test.cpu()


plt.subplot(1, 3, 1)
plt.pcolormesh(T_vis_cpu, X_cpu, prob_density_cpu, shading = 'gouraud', cmap = 'viridis')
plt.colorbar(label = 'ψψ*'); plt.xlabel('Time (t)') ; plt.ylabel('Position (x)') ; plt.title('Learned Probability Density')
plt.subplot(1,3,2)
t_slices = [0.0, 0.5, 1.0, 1.5, 2.0]
for t_val in t_slices:
    t_idx = np.argmin(np.abs(t_test_cpu.numpy() - t_val))
    plt.plot(x_test_cpu.numpy().squeeze(), prob_density_cpu[t_idx, :].numpy(), label =f't= {t_test[t_idx]: .1f}')
plt.xlabel('Position (x)') ; plt.ylabel("Probability Density: ψψ*") ; plt.title('Time Snapshots') ; plt.legend() ; plt.grid(True)
plt.subplot(1, 3, 3)
plt.plot(np.arange(len(energy_history)) * 1000, energy_history, 'o-')
plt.xlabel('Epoch') ; plt.ylabel('Expectation Value: <H>') ; plt.title('Energy Conservation') ; plt.grid(True)
#plt.tight_layout()
plt.show()


#10. Symbolic Regression with PySR
try:
    from pysr import PySRRegressor
except ImportError:
    print('\nPySR is not installed')
else:
    print("\nUsing PySR to Find the Wave Packet's Trajectory")
    model.eval()

    time_points = t_test_cpu.squeeze().numpy()
    peak_positions = []
    prob_density_numpy = prob_density_cpu.numpy()

    for i in range(len(time_points)):
        peak_index = np.argmax(prob_density_numpy[i, :])
        peak_x = x_test[peak_index].item()
        peak_positions.append(peak_x)

    X_pysr = time_points.reshape(-1, 1)
    y_pysr = np.array(peak_positions)

    pysr_model = PySRRegressor(

        niterations = 50,
        binary_operators = ["+", "-", "*", "/"],
        unary_operators = ["sqrt"],
        model_selection = "best",
    )

    print("Fitting PySR model to find x_peak(r)...")
    pysr_model.fit(X_pysr, y_pysr)

    print("\n\n\nPySR discoverd following equations")
    print(pysr_model)


#11. Use PySINDy to Find the Governing Differential Equation

try:
    import pysindy as ps
except ImportError:
    print("\nPySINDy not installed. Skipping SINDy analysis. To install: pip install pysindy")
else:
    print("\n--- Running Optimized SINDy to Rediscover the PDE ---")
    model.eval()

    # 1. Prepare data for SINDy (complex-valued) at t=1.5
    t_sindy = torch.full_like(x_grid, 1.5, requires_grad=True)
    psi_real_sindy, psi_imag_sindy = model(x_grid, t_sindy)

    d_real_dt = grad(psi_real_sindy.sum(), t_sindy, create_graph=True, retain_graph=True)[0]
    d_imag_dt = grad(psi_imag_sindy.sum(), t_sindy, create_graph=True, retain_graph=True)[0]

    psi_real_np = psi_real_sindy.cpu().detach().numpy()
    psi_imag_np = psi_imag_sindy.cpu().detach().numpy()
    d_real_dt_np = d_real_dt.cpu().detach().numpy()
    d_imag_dt_np = d_imag_dt.cpu().detach().numpy()

    X_sindy = np.column_stack([psi_real_np, psi_imag_np])
    X_dot_sindy = np.column_stack([d_real_dt_np, d_imag_dt_np])


    # 2. Define the robust, complex-valued custom library
    def salpeter_op_complex(psi_data):

        psi_data = np.asarray(psi_data)

        print(f"Result output shape: {psi_data.shape}")

        n_samples = psi_data.shape[0]
        result = np.empty((n_samples, 2))

        for i in range(n_samples):
            psi_real = psi_data[i, 0].reshape(1, -1)
            psi_imag = psi_data[i, 1].reshape(1, -1)

            rhs_real, rhs_imag = apply_sqrt_operator_fft(
                torch.from_numpy(psi_real).float(),
                torch.from_numpy(psi_imag).float(),
                x_domain, hbar, m, c
            )

            result[i, 0] = rhs_real.item()
            result[i, 1] = rhs_imag.item()

        print(f"Result output shape: {result.shape}")

        return result

    def real_wrapper(x_real):
        """Process real part only"""
        dummy_imag = np.zeros_like(x_real)
        combined = np.column_stack([x_real, dummy_imag])
        return salpeter_op_complex(combined)[:, 0]

    def imag_wrapper(x_imag):
        """Process imaginary part only"""
        dummy_real = np.zeros_like(x_imag)
        combined = np.column_stack([dummy_real, x_imag])
        return salpeter_op_complex(combined)[:, 1]


    custom_library = ps.CustomLibrary(
        library_functions=[
            lambda x: x,
            real_wrapper,
            imag_wrapper
        ],
        function_names=[
            lambda x: x,
            lambda x: f'H({x})_real',
            lambda x: f'H({x})_imag'
        ]
    )

    print(f"X_sindy shape: {X_sindy.shape}")
    print(f"X_dot_sindy shape: {X_dot_sindy.shape}")

    # 3. Instantiate and run SINDy
    sindy_model = ps.SINDy(
        feature_names=["ψ_real", "ψ_imag"],
       optimizer = ps.STLSQ(
                            threshold=0.02,  # More sensitive term detection
                            alpha=0.1,       # Moderate regularization
                            max_iter=100     # Ensure convergence
                    ),
        feature_library = custom_library
    )


    sindy_model.fit(X_sindy, x_dot = X_dot_sindy, t= 1.5)

    test_output = custom_library.transform(X_sindy[:1])
    print(f"Library output shape: {test_output.shape}")


    # 4. Print results
    print("\nSINDy discovered the following equations:")
    print("For real part:")
    sindy_model.print(precision=3)

    print("\nExpected equation for ħ=1:")
    print("dψ_real/dt = Im[H(ψ)]")
    print("dψ_imag/dt = -Re[H(ψ)]")
    print("Which together represent dψ/dt = -i*H(ψ)")

print("\n--- End of Program ---")

In [ ]:
def plot_nn_prediction(model, x_domain = (-10, 10), time_points = [0.0, 0.5, 1.0, 1.5, 2.0], save_path = None):

    device = next(model.parameters()).device
    x_grid = torch.linspace(x_domain[0], x_domain[1], 500).view(-1, 1).to(device)
    results = []

    for t in time_points:
        t_grid = torch.full_like(x_grid, t)
        with torch.no_grad():

            psi_r, psi_i = model(x_grid, t_grid)

        psi_r_cpu = psi_r.cpu()
        psi_i_cpu = psi_i.cpu()
        prob_cpu = (psi_i_cpu**2 + psi_i_cpu**2)

        df = pd.DataFrame({
            'Position' : x_grid.cpu().numpy().flatten(),
            'ψ_real' : psi_r_cpu.numpy().flatten(),
            'ψ_imag': psi_i_cpu.numpy().flatten(),
            'Probability' : prob_cpu.numpy().flatten(),
            'Time' : f"t = {t:.1f}"
        })
        results.append(df)
    plot_df = pd.concat(results)

    sns.set_style("whitegrid")
    plt.rcParams['font.family'] = 'serif'
    plt.rcParams['axes.labelsize'] = 12
    fig = plt.figure(figsize = (14, 8), dpi = 300)

    # Wavefunction component plot
    ax1 = plt.subplot2grid((2, 2), (0, 0), colspan = 1)
    melt_df = plot_df.melt(id_vars = ['Position', 'Time'],
                           value_vars = ['ψ_real', 'ψ_imag'],
                           var_name = 'Component')

    sns.lineplot(data = melt_df, x = 'Position', y = 'value',
                hue = 'Time', style = 'Component',
                palette = 'viridis', linewidth = 1.2, ax = ax1)
    ax1.set_title('(a) Wavefunction Evolution', pad = 10)
    ax1.set_ylabel('Amplitude')
    ax1.axhline(0, color = 'black', linestyle = ':', alpha = 0.3)
    ax1.axvline(0, color='black', linestyle=':', alpha=0.3)
    ax1.legend(title=None, frameon=False, bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.subplots_adjust(right=0.8)

    # Probability Destiny Plot
    ax2 = plt.subplot2grid((2, 2), (0, 1), colspan = 1)
    sns.lineplot(data = plot_df, x = 'Position', y = 'Probability',
                hue = 'Time', palette = 'viridis',
                linewidth = 1.5, ax = ax2)
    ax2.set_title("(b) Probability Density Evolution")
    ax2.set_ylabel("|ψ(x,t)|²")
    ax2.legend(title = None, frameon = False)

    # Phase space trajectory
    ax3 = plt.subplot2grid((2, 2), (1, 0), colspan = 1)
    sns.scatterplot(data = plot_df, x = 'ψ_real', y = 'ψ_imag',
                hue = 'Time', palette = 'viridis',
                size = 'Probability', sizes = (10, 200),
                alpha = 0.7, ax = ax3)

    ax3.set_title("(c) Phase Space Trajectory", pad=10)
    ax3.set_xlabel("Re(ψ)")
    ax3.set_ylabel("Im(ψ)")
    ax3.axhline(0, color='black', linestyle=':', alpha=0.3)
    ax3.axvline(0, color='black', linestyle=':', alpha=0.3)
    ax3.legend(title=None, frameon=False, bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.subplots_adjust(right=0.8)
    # Time slices heatmap
    ax4 = plt.subplot2grid((2, 2), (1, 1), colspan=1)
    pivot_df = plot_df.pivot(index='Position', columns='Time', values='Probability')
    sns.heatmap(pivot_df, cmap="plasma", cbar_kws={'label': '|ψ|²'}, ax=ax4)
    ax4.set_title("(d) Temporal Evolution", pad=10)
    ax4.set_xlabel("Time")
    ax4.set_ylabel("Position")

    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, bbox_inches='tight', dpi=300)
    plt.show()

if __name__ == "__main__":

    plot_nn_prediction(
        model=model,
        x_domain=(-15, 15),
        time_points=np.linspace(0, 2, 5),
        save_path="pinn_predictions.png"
    )

In [ ]:

# Data Preparation

pos_numpy = x_grid.squeeze().cpu().numpy()
time_numpy = t_test.cpu().numpy()
k_numpy = 2 * np.pi * np.fft.fftfreq(pos_numpy.size, d=(pos_numpy[1]-pos_numpy[0]))
k_sort_indices, k_numpy_sorted = np.argsort(k_numpy), k_numpy[np.argsort(k_numpy)]

analysis_results, snapshot_indices = [], [0, 50, 99]
print("Calculating advanced properties for snapshots...")
device = next(model.parameters()).device


for i in snapshot_indices:
    t = time_numpy[i]
    t_tensor = torch.full_like(x_grid, t).to(device)

    with torch.no_grad():
        psi_real, psi_imag = model(x_grid.to(device), t_tensor)

    psi_x = (psi_real.squeeze().cpu().numpy() + 1j * psi_imag.squeeze().cpu().numpy())

    psi_k = np.fft.fft(psi_x, norm='ortho')
    prob_k = np.abs(psi_k)**2

    dx = pos_numpy[1]-pos_numpy[0]
    dk = k_numpy_sorted[1]-k_numpy_sorted[0]
    exp_x = np.sum(pos_numpy * np.abs(psi_x)**2) * dx
    exp_p = np.sum(k_numpy_sorted * prob_k[k_sort_indices]) * dk

    exp_x2 = np.sum((pos_numpy**2) * np.abs(psi_x)**2) * dx
    exp_p2 = np.sum((k_numpy_sorted**2) * prob_k[k_sort_indices]) * dk
    delta_x = np.sqrt(exp_x2 - exp_x**2)
    delta_p = np.sqrt(exp_p2 - exp_p**2)

    wigner = np.zeros((len(pos_numpy), len(pos_numpy)), dtype=np.float64)
    x_indices = np.arange(len(pos_numpy))

    for j_x in x_indices:
        y_conv = np.zeros(len(pos_numpy), dtype=np.complex128)
        for j_y in x_indices:
            idx_plus = (j_x + j_y) % len(pos_numpy)
            idx_minus = (j_x - j_y + len(pos_numpy)) % len(pos_numpy)
            y_conv[j_y] = psi_x[idx_plus] * np.conj(psi_x[idx_minus])
        wigner_slice = np.fft.fft(y_conv, norm='ortho')
        wigner[:, j_x] = np.real(np.fft.fftshift(wigner_slice))

    analysis_results.append({'time': t, '<x>': exp_x, '<p>': exp_p, 'Δx': delta_x, 'Δp': delta_p, 'ΔxΔp': delta_x * delta_p, 'prob_k': prob_k[k_sort_indices], 'wigner': wigner})

df_analysis = pd.DataFrame(analysis_results)

# Visualization Dashboard
sns.set_theme(style="whitegrid")
fig = plt.figure(figsize=(20, 13))
gs = fig.add_gridspec(2, 3)

ax1 = fig.add_subplot(gs[0, 0])
for idx, row in df_analysis.iterrows(): ax1.plot(k_numpy_sorted, row['prob_k'], label=f"t={row['time']:.1f}s")
ax1.set_title("Evolution in Momentum Space", fontweight='bold'); ax1.set_xlabel("Momentum (k)"); ax1.legend(); ax1.set_xlim(-15, 15); ax1.grid(True)

ax2 = fig.add_subplot(gs[0, 1])
ax2.plot(df_analysis['time'], df_analysis['ΔxΔp'], 'o-', color='crimson'); ax2.axhline(y=hbar/2, color='k', linestyle='--', label=f"Heisenberg Limit (ħ/2)")
ax2.set_title("Heisenberg Uncertainty Principle", fontweight='bold'); ax2.set_xlabel("Time (t)"); ax2.set_ylabel("ΔxΔp"); ax2.legend(); ax2.set_ylim(bottom=0.4); ax2.grid(True)

ax3 = fig.add_subplot(gs[0, 2])
trajectory_x = df_analysis['<x>']
group_velocity = df_analysis['<p>'][0] / np.sqrt(m**2 + df_analysis['<p>'][0]**2)
initial_pos = df_analysis['<x>'][0]
theoretical_traj = group_velocity * df_analysis['time'] + initial_pos
ax3.plot(df_analysis['time'], trajectory_x, 'o', ms=10, label='Actual Trajectory <x>(t)'); ax3.plot(df_analysis['time'], theoretical_traj, '--', lw=3, label='Classical Trajectory')
ax3.set_title("Ehrenfest Theorem Validation", fontweight='bold'); ax3.set_xlabel("Time (t)"); ax3.set_ylabel("Position <x>"); ax3.legend(); ax3.grid(True)

wigner_axes = [fig.add_subplot(gs[1, i]) for i in range(3)]
max_wigner_val = np.max(np.abs(df_analysis.loc[0, 'wigner'])) * 0.5

for i, ax in enumerate(wigner_axes):
    wigner_matrix = df_analysis.loc[i, 'wigner']
    time_val = df_analysis.loc[i, 'time']

    im = ax.pcolormesh(pos_numpy, k_numpy_sorted, wigner_matrix, cmap='seismic',
                       vmin=-max_wigner_val, vmax=max_wigner_val, shading='gouraud')

    ax.set_title(f"Wigner Function at t={time_val:.1f}s", fontweight='bold')
    ax.set_xlabel("Position (x)")
    if i == 0:
        ax.set_ylabel("Momentum (k)")
    ax.set_ylim(-15, 15)

fig.suptitle("A Deeper Look at the Quantum Wave Packet", fontsize=24)
fig.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()